In [1]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.utils as vutils

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [3]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done")

Done


In [4]:
from Scripts.utils import load_mnist_dataset

In [5]:
data_path = project_root / "data"
train_dataloader, test_dataloader = load_mnist_dataset(
    data_path=data_path,
    batch_size=128
)

In [6]:
next(iter(train_dataloader))

[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         ...,
 
 
         [[[-1., -1., -1.,  ..., -

In [7]:
class Generator(nn.Module):
    def __init__(self, noise_size: int):
        super().__init__()

        self.tanh   = nn.Tanh()
        self.relu   = nn.ReLU()
        self.layer1 = nn.Linear(in_features=noise_size, out_features=7*7*256)
        self.bn1    = nn.BatchNorm2d(num_features=256)
        self.layer2 = nn.ConvTranspose2d(in_channels=256, out_channels=128, kernel_size=4, stride=2, padding=1)
        self.bn2    = nn.BatchNorm2d(num_features=128)
        self.layer3 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=4, stride=2, padding=1)
        self.bn3    = nn.BatchNorm2d(num_features=64)
        self.layer4 = nn.Conv2d(in_channels=64, out_channels=1, kernel_size=3, stride=1, padding=1)

    def forward(self, X):

        X = self.layer1(X)
        X = X.view(X.shape[0], 256, 7, 7)       # Shape: (batch_size, 256, 7, 7)
        X = self.bn1(X)                         # Shape: (batch_size, 256, 7, 7)
        X = self.relu(X)                        # Shape: (batch_size, 256, 7, 7)
        X = self.layer2(X)                      # Shape: (batch_size, 128, 14, 14)
        X = self.bn2(X)                         # Shape: (batch_size, 128, 14, 14)
        X = self.relu(X)                        # Shape: (batch_size, 128, 14, 14)
        X = self.layer3(X)                      # Shape: (batch_size, 64, 28, 28)
        X = self.bn3(X)                         # Shape: (batch_size, 64, 28, 28)
        X = self.relu(X)                        # Shape: (batch_size, 64, 28, 28)
        X = self.layer4(X)                      # Shape: (batch_size, 1, 28, 28)
        X = self.tanh(X)

        return X

In [8]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.lrelu  = nn.LeakyReLU(0.2)
        self.layer1 = nn.Conv2d(1, 128, 5, stride=2, padding=2)
        self.layer2 = nn.Conv2d(128, 256, 5, stride=2, padding=2)
        self.bn1    = nn.BatchNorm2d(256)
        self.layer3 = nn.Linear(256*7*7, 1)

    def forward(self, X):
        X = self.lrelu(self.layer1(X))
        X = self.bn1(self.layer2(X))
        X = self.lrelu(X)
        X = X.view(X.shape[0], -1)
        X = self.layer3(X)
        return X

In [9]:
def visualize_generated_digits(generator, noise_dim, device, num_images=16):
    """Generates and displays a grid of images from the current generator state."""
    
    # 1. Set generator to evaluation mode (disables dropout, fixes batchnorm stats)
    generator.eval()
    
    # 2. Generate images without tracking gradients (saves memory/compute)
    with torch.no_grad():
        # Create fresh noise
        noise = torch.randn(num_images, noise_dim, 1, 1, device=device)
        # Generate raw fake images
        fake_images = generator(noise)
        
    # 3. Un-normalize the images from [-1, 1] back to [0, 1]
    fake_images = (fake_images + 1) / 2.0
    
    # 4. Arrange the batch of images into a single grid image
    # nrow=4 means a 4x4 grid for 16 images
    grid = vutils.make_grid(fake_images, nrow=4, padding=2, normalize=False)
    
    # 5. Convert to numpy, move to CPU, and rearrange dimensions for matplotlib
    # PyTorch is (C, H, W) -> Matplotlib needs (H, W, C)
    grid_np = grid.cpu().numpy().transpose((1, 2, 0))
    
    # 6. Plot the grid
    plt.figure(figsize=(6, 6))
    plt.axis("off")
    plt.title("WGAN Generated Digits")
    # cmap='gray' is required otherwise matplotlib tries to add artificial colors to 1-channel images
    plt.imshow(grid_np, cmap='gray') 
    plt.show()
    
    # 7. Crucial: Put the generator back into training mode!
    generator.train()

In [10]:
def train(
        data_loader,
        generator: Generator,
        discriminator: Discriminator,
        a: int,
        b: int,
        c: int,
        opt_gen: optim.Adam,
        opt_dis: optim.Adam,
        epochs: int,
        noise_dim: int,
        device
):
    # training mode
    generator.train()
    discriminator.train()

    # loss track
    generator_cost = []
    discriminator_cost = []

    for epoch in range(epochs):
        total_discriminator_loss = 0
        total_generator_loss = 0
        for batch, _ in data_loader:
            batch = batch.to(device)

            # phase 1 training the discriminator
            noise = torch.randn(size=[batch.shape[0], noise_dim], device=device)
            fake_images = generator(noise).detach()
            real_scores = discriminator(batch)
            fake_scores = discriminator(fake_images)

            # discrimator cost
            disc_loss = 0.5*torch.mean((real_scores - torch.ones_like(real_scores)*b)**2) + 0.5*torch.mean((fake_scores - torch.ones_like(fake_scores)*a)**2)
            total_discriminator_loss += disc_loss.item()

            # backprop and optim
            opt_dis.zero_grad()
            disc_loss.backward()
            opt_dis.step()

            # phase 2 training the generator
            new_noise = torch.randn(size=[batch.shape[0], noise_dim], device=device)
            new_fake_images = generator(new_noise)
            new_fake_scores = discriminator(new_fake_images)

            # generator cost
            generate_loss = 0.5*torch.mean((new_fake_scores - torch.ones_like(new_fake_scores)*c)**2)
            total_generator_loss += generate_loss.item()

            # backprop and optim
            opt_gen.zero_grad()
            generate_loss.backward()
            opt_gen.step()

        avg_disc_batch_loss = total_discriminator_loss/len(data_loader)
        avg_gene_batch_loss = total_generator_loss/len(data_loader)
        discriminator_cost.append(avg_disc_batch_loss)
        generator_cost.append(avg_gene_batch_loss)

        # check the generation my the generator
        visualize_generated_digits(
            generator=generator,
            noise_dim=noise_dim,
            device=device
        )

        print(f"Epoch: {epoch+1}/{epochs} | Avg_Generator_loss_per_batch: {avg_gene_batch_loss:.4f} | Avg_Discriminator_loss_per_batch: {avg_disc_batch_loss:.4f}")

    return generator, discriminator, generator_cost, discriminator_cost

In [11]:
NOISE_DIM = 100
LR = 1e-4
A = -1
B = 1
C = 1
EPOCHS = 100
generator = Generator(noise_size=NOISE_DIM).to(device=device)
discriminator = Discriminator().to(device=device)
opt_gen = optim.Adam(params=generator.parameters(), lr=LR, betas=(0.5, 0.999))
opt_dis = optim.Adam(params=discriminator.parameters(), lr=LR, betas=(0.5, 0.999))

In [12]:
trained_generator, trained_discriminator, generator_cost_track, discriminator_loss_track = train(
    data_loader=train_dataloader,
    generator=generator,
    discriminator=discriminator,
    a=A,
    b=B,
    c=C,
    opt_gen=opt_gen,
    opt_dis=opt_dis,
    epochs=EPOCHS,
    noise_dim=NOISE_DIM,
    device=device
)

KeyboardInterrupt: 